# 2.7 元学习 (Meta-Learning)

学习如何学习，使模型能快速适应新任务，是少样本学习的理论基础。

本节涵盖：
- MAML
- 上下文学习（In-Context Learning）
- 提示学习
- 基于任务的元训练

## 1. MAML（Model-Agnostic Meta-Learning）

**目的**：寻找对任务变化敏感的初始化参数

**基本原理**：通过双层优化：内层在任务上梯度更新得到任务特定参数，外层优化初始参数使内层更新后的参数在任务上表现最优，使模型只需少量梯度步即可适应新任务。

**MAML算法**：
1. 采样任务Ti
2. 内循环：在Ti的支持集上用梯度下降得到适应参数θ'i = θ - α∇L(θ)
3. 外循环：在Ti的查询集上计算L(θ'i)，对θ求梯度更新θ
4. 目标：min_θ Σ L(θ - α∇L(θ))——优化初始化使一步适应后的损失最小

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.func import functional_call

torch.manual_seed(42)

class MAMLModel(nn.Module):
    def __init__(self, input_dim=5, hidden=32, output_dim=5):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden), nn.ReLU(),
            nn.Linear(hidden, hidden), nn.ReLU(),
            nn.Linear(hidden, output_dim)
        )
    def forward(self, x, params=None):
        if params is None:
            return self.net(x)
        return functional_call(self.net, params, x)

def generate_task(n_way=5, n_support=10, n_query=10, input_dim=5):
    W = torch.randn(n_way, input_dim)
    X_support = torch.randn(n_support * n_way, input_dim)
    y_support = torch.arange(n_way).repeat(n_support)
    X_support = X_support + W[y_support] * 0.3
    X_query = torch.randn(n_query * n_way, input_dim)
    y_query = torch.arange(n_way).repeat(n_query)
    X_query = X_query + W[y_query] * 0.3
    return X_support, y_support, X_query, y_query

model = MAMLModel()
meta_optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
inner_lr = 0.01

print('=== MAML (Model-Agnostic Meta-Learning) ===')
for meta_step in range(30):
    meta_loss = 0.0
    n_tasks = 4
    for _ in range(n_tasks):
        X_s, y_s, X_q, y_q = generate_task()

        support_logits = model(X_s)
        support_loss = F.cross_entropy(support_logits, y_s)
        grads = torch.autograd.grad(support_loss, model.parameters(), create_graph=True)
        fast_params = {name: p - inner_lr * g for (name, p), g in zip(model.named_parameters(), grads)}

        query_logits = model(X_q, params=fast_params)
        query_loss = F.cross_entropy(query_logits, y_q)
        meta_loss += query_loss

    meta_loss /= n_tasks
    meta_optimizer.zero_grad()
    meta_loss.backward()
    meta_optimizer.step()

    if (meta_step + 1) % 10 == 0:
        print(f'Meta-step {meta_step+1}: meta_loss={meta_loss.item():.4f}')

print('\nKey: MAML finds initialization that quickly adapts to new tasks with few gradient steps.')
print('Inner loop: adapt on support set. Outer loop: optimize initialization on query set.')

## 2. 上下文学习（In-Context Learning）

**目的**：通过提示中的示例快速适应新任务

**基本原理**：不更新任何参数，仅在输入提示中提供几个示例，模型通过注意力机制从示例中学习任务模式并泛化到新输入。这是大模型涌现出的元学习能力。

**ICL的工作机制**：
- 示例作为键值对存储在注意力中
- 新输入通过注意力检索相关示例
- 模型根据检索到的示例模式生成输出
- 本质上是一种隐式的梯度下降（Akyurek et al., 2022）

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math

torch.manual_seed(42)

class SimpleICLModel(nn.Module):
    def __init__(self, d_model=64, n_heads=4, vocab_size=50):
        super().__init__()
        self.d_model = d_model
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.attn = nn.MultiheadAttention(d_model, n_heads, batch_first=True)
        self.output = nn.Linear(d_model, vocab_size)

    def forward(self, context_tokens, query_token):
        context_emb = self.embedding(context_tokens)
        query_emb = self.embedding(query_token).unsqueeze(1)
        full_emb = torch.cat([context_emb, query_emb], dim=1)
        causal_mask = nn.Transformer.generate_square_subsequent_mask(full_emb.size(1))
        attn_out, attn_weights = self.attn(full_emb, full_emb, full_emb, attn_mask=causal_mask)
        query_output = attn_out[:, -1, :]
        return self.output(query_output), attn_weights[:, -1, :]

model = SimpleICLModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def make_icl_task(vocab_size=50, n_examples=3, n_tasks_per_batch=16):
    input_tokens = torch.randint(0, vocab_size // 2, (n_tasks_per_batch, n_examples))
    output_tokens = input_tokens + vocab_size // 2
    context = torch.stack([input_tokens, output_tokens], dim=-1).reshape(n_tasks_per_batch, -1)
    query_input = torch.randint(0, vocab_size // 2, (n_tasks_per_batch,))
    query_target = query_input + vocab_size // 2
    return context, query_input, query_target

print('=== In-Context Learning Simulation ===')
for epoch in range(30):
    context, query_input, query_target = make_icl_task()
    logits, attn_w = model(context, query_input)
    loss = F.cross_entropy(logits, query_target)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        acc = (logits.argmax(-1) == query_target).float().mean()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, icl_accuracy={acc:.3f}')

with torch.no_grad():
    context, query_input, query_target = make_icl_task(n_examples=5)
    logits, attn_w = model(context, query_input)
    pred = logits.argmax(-1)
    acc = (pred == query_target).float().mean()
    print(f'\nTest with 5 examples: accuracy={acc:.3f}')
    for i in range(3):
        print(f'  Input: {query_input[i].item()}, Target: {query_target[i].item()}, Pred: {pred[i].item()}')

print('\nKey: ICL learns from examples in the prompt without any parameter updates.')
print('More examples -> better accuracy (up to context length limit).')

## 3. 提示学习

**目的**：将任务适配问题转化为提示优化问题

**基本原理**：通过优化连续提示向量（Prompt Tuning）或搜索离散提示文本（APE），使预训练模型在不修改参数的情况下适配新任务。

**提示学习 vs 传统微调**：
- 传统微调：修改模型所有参数
- Prompt Tuning：仅优化输入端的连续向量，参数量极少
- Prefix Tuning：在每层添加可学习前缀，效果更好但参数更多

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(42)

class PromptTuningModel(nn.Module):
    def __init__(self, d_model=64, vocab_size=100, n_prompt_tokens=5, n_classes=3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, d_model)
        self.encoder = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model, nhead=4, batch_first=True),
            num_layers=2
        )
        self.prompt_embeddings = nn.Parameter(torch.randn(n_prompt_tokens, d_model) * 0.02)
        self.classifier = nn.Linear(d_model, n_classes)
        self.n_prompt_tokens = n_prompt_tokens

    def forward(self, input_ids):
        input_emb = self.embedding(input_ids)
        batch_size = input_emb.size(0)
        prompt_emb = self.prompt_embeddings.unsqueeze(0).expand(batch_size, -1, -1)
        combined = torch.cat([prompt_emb, input_emb], dim=1)
        encoded = self.encoder(combined)
        cls_repr = encoded[:, self.n_prompt_tokens, :]
        return self.classifier(cls_repr)

model = PromptTuningModel()
for p in model.parameters():
    p.requires_grad = False
model.prompt_embeddings.requires_grad = True
model.classifier.requires_grad = True

optimizer = torch.optim.Adam(
    [model.prompt_embeddings, model.classifier.weight, model.classifier.bias], lr=1e-3
)

n_samples = 200
X = torch.randint(0, 100, (n_samples, 10))
y = torch.randint(0, 3, (n_samples,))

total_params = sum(p.numel() for p in model.parameters())
tunable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print('=== Prompt Tuning Demo ===')
print(f'Total parameters: {total_params:,}')
print(f'Tunable parameters: {tunable_params:,} ({tunable_params/total_params*100:.1f}%)')

for epoch in range(30):
    logits = model(X)
    loss = F.cross_entropy(logits, y)
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    if (epoch+1) % 10 == 0:
        acc = (logits.argmax(-1) == y).float().mean()
        print(f'Epoch {epoch+1}: loss={loss.item():.4f}, acc={acc:.3f}')

print('\nKey: Only prompt embeddings + classifier are trained, model backbone is frozen.')
print('This is extremely parameter-efficient for adapting large models.')

## 4. 基于任务的元训练

**目的**：在大量任务上训练以获得快速适应能力

**基本原理**：构建大量不同的任务，使模型学会从少量示例中快速推断任务意图并正确执行。代表：MetaICL在大量任务上进行元训练，使模型学会如何从上下文中学习。

**元训练 vs 普通训练**：
- 普通训练：在数据上训练，学数据模式
- 元训练：在任务上训练，学如何快速适应新任务
- 元训练的泛化目标是新任务，而非新数据

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import random

torch.manual_seed(42)
random.seed(42)

class TaskFamily:
    def __init__(self, name, input_dim, output_dim, n_classes):
        self.name = name
        self.input_dim = input_dim
        self.output_dim = output_dim
        self.n_classes = n_classes

task_families = [
    TaskFamily('sentiment', 10, 3, 3),
    TaskFamily('topic', 10, 5, 5),
    TaskFamily('spam', 10, 2, 2),
    TaskFamily('language', 10, 4, 4),
]

class MetaTrainedModel(nn.Module):
    def __init__(self, input_dim=10, hidden=32, max_classes=5):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(input_dim, hidden), nn.ReLU(), nn.Linear(hidden, hidden))
        self.head = nn.Linear(hidden, max_classes)
    def forward(self, x):
        return self.head(self.encoder(x))

def sample_task(family, n_support=5, n_query=10):
    W = torch.randn(family.n_classes, family.input_dim)
    X_s = torch.randn(n_support * family.n_classes, family.input_dim) + W[torch.arange(family.n_classes).repeat(n_support)] * 0.5
    y_s = torch.arange(family.n_classes).repeat(n_support)
    X_q = torch.randn(n_query * family.n_classes, family.input_dim) + W[torch.arange(family.n_classes).repeat(n_query)] * 0.5
    y_q = torch.arange(family.n_classes).repeat(n_query)
    return X_s, y_s, X_q, y_q, family.n_classes

model = MetaTrainedModel()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

print('=== Meta-Training on Multiple Task Families ===')
for meta_epoch in range(30):
    total_loss = 0
    total_acc = 0
    for family in task_families:
        X_s, y_s, X_q, y_q, n_cls = sample_task(family)
        with torch.no_grad():
            support_logits = model(X_s)[:, :n_cls]
            support_loss = F.cross_entropy(support_logits, y_s)
            grads = torch.autograd.grad(support_loss, model.parameters(), retain_graph=False)
            fast_params = {name: p - 0.01 * g for (name, p), g in zip(model.named_parameters(), grads)}

        query_logits = model(X_q)[:, :n_cls]
        loss = F.cross_entropy(query_logits, y_q)
        total_loss += loss.item()
        total_acc += (query_logits.argmax(-1) == y_q).float().mean().item()

    total_loss /= len(task_families)
    total_acc /= len(task_families)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()

    if (meta_epoch + 1) % 10 == 0:
        print(f'Meta-epoch {meta_epoch+1}: loss={total_loss:.4f}, avg_acc={total_acc:.3f}')

print('\nKey: Meta-training across diverse tasks teaches the model to quickly adapt.')
print('After meta-training, the model can learn new tasks from just a few examples.')